# `search_api` Buyer Notebook — paying for a GET (Deno/TypeScript)

Drives `search_api.ts`'s two paid routes as a real client would: `GET /fetch` reads one web page,
`GET /search` proxies Brave's LLM Context API. Third of this directory's `_buyer` notebooks, after
`genimg_x402_buyer.ipynb` (scheme `exact`) and `sc_llm_x402_buyer.ipynb` (scheme
`batch-settlement`), and it only ever talks to the resource server over plain HTTP via
`wrapFetchWithPayment` — the server advertises everything else.

Three things here are not covered by `test/search_api.test.ts`, which mocks the SDK:

1. **A GET can be paid for at all.** Both of the other paid endpoints are POSTs, and the payment
   arrives in a header either way — there is no body to fall back on.
2. **The 402 this handler emits is readable by `@x402/fetch`**, prices and all.
3. **The second paid call opens no second channel.** That is the whole design: a tool call bills
   onto the channel the chat already funded.

## ⚠️ This notebook always spends real money

There is no testnet option. `/search` and `/fetch` spend on the caller's behalf — Brave bills per
query, `/fetch` is egress — so testnet USDC, which is free, would buy a metered API for nothing.
`MAINNET_NETWORKS` in `search_api.ts` deliberately lists only Optimism and Base mainnet.

A full run costs **$0.011** (one fetch at $0.001, one search at $0.01) plus deposit gas, out of a
$0.10 channel deposit. The rest of the deposit stays escrowed until `llmx402cron` claims it
(within 12h) or a refund after the 24h `withdrawDelay` — it is not spent, but it is not immediately
back in your wallet either.

## Prerequisites

1. **Deno Jupyter kernel** — `deno jupyter --install` (the same global kernel the sibling notebooks
   use).
2. **`scw_js/.env`** needs:
   - `TEST_WALLET_PRIVATE_KEY` — the buyer wallet, funded with **mainnet** USDC on Base or
     Optimism, plus a little ETH on that chain for the deposit tx. Notebook-testing convenience
     only; no scw_js production code reads this key.
   - `NFT_WALLET_PUBLIC_KEY` — the receiver, shared with the chat endpoint. This is what makes the
     channel the same one.
   - `RECEIVER_AUTHORIZER_PRIVATE_KEY` — needed for `createLLMResourceServer()` to construct at
     all. A pure off-chain signer, no funding.
   - `BRAVE_API_KEY` — read by the **server**, not by this notebook.
3. **The server**, either locally (`cd scw_js && npm run dev:search`, port 8084) or deployed. The
   `USE_DEPLOYED` flag below switches between them, and the channel is the same either way: it is
   on-chain state keyed by (buyer, receiver, network, asset), not anything a server holds.

In [1]:
// Setup: imports + env
import { load } from "https://deno.land/std@0.224.0/dotenv/mod.ts";
import { privateKeyToAccount } from "npm:viem@2/accounts";

// scw_js's own .env, one level up — the single source of truth (buyer key, receiver address, and
// the server's own secrets all live here; the server reads the same file).
const env = await load({ envPath: "../.env", examplePath: null, export: true });

const PRIVATE_KEY = env.TEST_WALLET_PRIVATE_KEY;
if (!PRIVATE_KEY) {
    throw new Error(
        "TEST_WALLET_PRIVATE_KEY missing from scw_js/.env — add a key funded with MAINNET USDC " +
        "(Base or Optimism). There is no testnet path for these routes; see the cell above.",
    );
}
const account = privateKeyToAccount(`0x${PRIVATE_KEY.replace(/^0x/, "")}`);

console.log("🔎 search_api buyer notebook");
console.log(`   Buyer (payer): ${account.address}`);
console.log(`   Receiver     : ${env.NFT_WALLET_PUBLIC_KEY ?? "(not set — server will 500)"}`);

🔎 search_api buyer notebook
   Buyer (payer): 0x553179556FC2A39e535D65b921e01fA995E79101
   Receiver     : 0xAAEBC1441323B8ad6Bdf6793A8428166b510239C


## Network selection

Base or Optimism, both mainnet — see the warning above for why there is no third option. The two
differ in one way that has bitten this repo before: **USDC's EIP-712 domain name is `USD Coin` on
mainnet and `USDC` on testnet**, so a config copied from a testnet notebook signs vouchers no
server will accept. The table in `scw_js/README.md` → _Known USDC Domain Names_ is the verified
source.

**Base also offers EURC**, listed before USDC in the 402 (`scw_js/stablecoin_pricing.ts`) — so the
default `USE_BASE = true` pays EURC once the buyer-setup cell allows it. Optimism has no EURC
deployment. Search and fetch have their own EURC prices (`PRICE_ATOMIC` in `search_schemas.ts`);
nothing is converted from USDC.

`USE_DEPLOYED` is orthogonal to the network: same code path either way.

In [2]:
import { base, optimism } from "npm:viem@2/chains";

const USE_BASE = true; // false = Optimism mainnet

const NETWORK_CONFIG = {
    "base-mainnet": {
        caip2Network: "eip155:8453" as const, networkName: "Base Mainnet",
        chain: base, usdcName: "USD Coin",
        usdcAddress: "0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913" as `0x${string}`,
        // EURC on Base — see shared/chain-utils/src/addresses.ts and scw_js/README.md's EURC
        // domain-name table.
        eurcAddress: "0x60a3E35Cc302bFA44Cb288Bc5a4F316Fdb1adb42" as `0x${string}` | undefined,
        rpcUrl: "https://base-rpc.publicnode.com",
        explorer: "https://basescan.org",
    },
    "optimism-mainnet": {
        caip2Network: "eip155:10" as const, networkName: "Optimism Mainnet",
        chain: optimism, usdcName: "USD Coin",
        usdcAddress: "0x0b2C639c533813f4Aa9D7837CAf62653d097Ff85" as `0x${string}`,
        // No EURC on Optimism (Circle has no deployment there).
        eurcAddress: undefined as `0x${string}` | undefined,
        rpcUrl: "https://optimism-rpc.publicnode.com",
        explorer: "https://optimistic.etherscan.io",
    },
};
// RPCs pinned to publicnode rather than viem's bare defaults (mainnet.base.org /
// mainnet.optimism.io): opening a channel does a Multicall3 read batch, which those defaults have
// already rate-limited in this repo — surfacing as a generic ..._deposit_transaction_failed. Same
// endpoints website/wagmi.config.ts settled on.
const config = NETWORK_CONFIG[USE_BASE ? "base-mainnet" : "optimism-mainnet"];
const NETWORK = config.caip2Network;

// Local: `cd scw_js && npm run dev:search` (port 8084, set in search_api.ts's dev server block).
const USE_DEPLOYED = false;
const SERVICE_URL = USE_DEPLOYED
    ? "https://web-agent.fretchen.eu"
    : "http://localhost:8084";

console.log(`🚨 REAL MONEY on ${config.networkName}`);
console.log(`   ${NETWORK} • USDC ${config.usdcName} @ ${config.usdcAddress}`);
if (config.eurcAddress) {
    console.log(`   ${NETWORK} • EURC @ ${config.eurcAddress} (server prefers this over USDC)`);
}
console.log(`   Service: ${SERVICE_URL}`);

🚨 REAL MONEY on Base Mainnet
   eip155:8453 • USDC USD Coin @ 0x833589fCD6eDb6E08f4c7C32D4f71b54bdA02913
   Service: http://localhost:8084


## Buyer setup

Identical in shape to `sc_llm_x402_buyer.ipynb`'s buyer cell, and for the same reason: the scheme
is `batch-settlement`, so the client needs a channel store, a deposit strategy and a signer that
can read contracts. The one difference is the deposit floor, which is smaller here — these calls
cost a tenth of a cent, not three tenths.

In [3]:
import { x402Client, wrapFetchWithPayment } from "npm:@x402/fetch@^2.20.0";
import { toClientEvmSigner } from "npm:@x402/evm@^2.20.0";
import {
    BatchSettlementEvmScheme,
    type ClientChannelStorage,
    type BatchSettlementClientContext,
    type BatchSettlementDepositStrategyContext,
} from "npm:@x402/evm@^2.20.0/batch-settlement/client";
import { createPublicClient, http } from "npm:viem@2";

// readContract is documented as "optional" on ClientEvmSigner, but batch-settlement's
// corrective-402 recovery (processCorrectivePaymentRequired) unconditionally needs it — both
// recoverFromSignature and recoverFromOnChainState bail out with `if (!deps.signer.readContract)
// return false`. Without it, a client/server cumulative desync (an old channel reused across
// kernel restarts) surfaces as a hard 402 instead of self-healing.
const publicClient = createPublicClient({ chain: config.chain, transport: http(config.rpcUrl) });
const buyerSigner = toClientEvmSigner(
    { address: account.address, signTypedData: (a: any) => account.signTypedData(a) },
    publicClient,
);

// Channel storage. Deno exposes a global `localStorage` that persists across kernel restarts,
// which is what lets a second run reuse the first run's channel instead of depositing again.
class WebStorageClientChannelStorage implements ClientChannelStorage {
    constructor(private backend: Storage, private prefix = "x402-channel:") {}
    private keyFor(key: string) { return `${this.prefix}${key.toLowerCase()}`; }
    async get(key: string) {
        const raw = this.backend.getItem(this.keyFor(key));
        return raw ? (JSON.parse(raw) as BatchSettlementClientContext) : undefined;
    }
    async set(key: string, context: BatchSettlementClientContext) {
        this.backend.setItem(this.keyFor(key), JSON.stringify(context));
    }
    async delete(key: string) { this.backend.removeItem(this.keyFor(key)); }
}
const buyerStorage = new WebStorageClientChannelStorage(localStorage);

// $0.10 of escrow: ~9 searches or ~100 fetches. Deliberately smaller than useX402Chat.ts's $0.50
// floor, because this is mainnet and a notebook — but note it is ESCROW, not spend. Whatever is
// not consumed stays locked until llmx402cron claims the channel (≤12h) or a refund after the 24h
// withdrawDelay. Going much lower just buys more top-up transactions.
const MINIMUM_DEPOSIT_ATOMIC = 100_000n;

function depositStrategy(context: BatchSettlementDepositStrategyContext): string {
    const required = BigInt(context.minimumDepositAmount);
    return (required > MINIMUM_DEPOSIT_ATOMIC ? required : MINIMUM_DEPOSIT_ATOMIC).toString();
}

const buyerScheme = new BatchSettlementEvmScheme(buyerSigner, {
    storage: buyerStorage,
    depositStrategy,
});

const client = new x402Client();
client.register(NETWORK, buyerScheme);

// USDC already passes the SDK's default spend-control asset registry on both networks this
// notebook uses, so it needs no explicit entry. EURC does not exist in that registry at all, so
// without this it would be silently dropped from Base's offer rather than ever paid.
if (config.eurcAddress) {
    client.setSpendControls({
        allowedAssets: [{ network: NETWORK, asset: config.eurcAddress }],
    });
}

const fetchWithPayment = wrapFetchWithPayment(fetch, client);

/** The settlement receipt the server attaches to a paid 200. `transaction: ""` means a voucher
 *  was committed off-chain — no tx, which is the point of batch settlement. */
function settlement(res: Response) {
    const header = res.headers.get("Payment-Response");
    return header
        ? (JSON.parse(atob(header)) as {
              transaction?: string;
              network?: string;
              extensions?: { facilitatorFees?: { info?: { asset?: string } } };
          })
        : null;
}

console.log("✅ buyer client ready, registered for", NETWORK);

✅ buyer client ready, registered for eip155:8453


## The 402 is the price list

Reading it costs nothing — no wallet, no payment, no channel. A bare `fetch` gets the terms in the
base64 `Payment-Required` header, which is how any client discovers what it is being asked for.

Note the two prices, and that the quote is the same whatever the query string says: an unpaid
request is answered with the price before its query is even validated, so a client probing for
terms is never told its query is malformed instead.

In [4]:
async function quote(route: "search" | "fetch") {
    const res = await fetch(`${SERVICE_URL}/${route}`);
    const header = res.headers.get("Payment-Required");
    // Nothing reads the body here — the terms are in the header — so release it rather than
    // leaving an open response behind on every kernel run.
    await res.body?.cancel();
    if (!header) throw new Error(`No Payment-Required header on ${res.status} — is the server running?`);
    return { status: res.status, terms: JSON.parse(atob(header)) };
}

for (const route of ["fetch", "search"] as const) {
    const { status, terms } = await quote(route);
    console.log(`\nGET /${route} → ${status}  "${terms.resource.description}"`);
    console.log(`   resource: ${terms.resource.url}`);
    for (const accept of terms.accepts) {
        const usd = (Number(accept.amount) / 1e6).toFixed(4);
        const asset = accept.extra?.name ?? accept.asset;
        console.log(
            `   ${accept.network.padEnd(13)} ${accept.scheme}  ${accept.amount} ${asset} atomic ($${usd})` +
            `  → ${accept.payTo}  lock ${accept.maxTimeoutSeconds}s`,
        );
    }
}


GET /fetch → 402  "The readable text of one public web page"
   resource: https://mypersonaljscloudivnad9dy-searchapi.functions.fnc.fr-par.scw.cloud/fetch
   eip155:10     batch-settlement  1000 atomic ($0.0010)  → 0xAAEBC1441323B8ad6Bdf6793A8428166b510239C  lock 30s
   eip155:8453   batch-settlement  1000 atomic ($0.0010)  → 0xAAEBC1441323B8ad6Bdf6793A8428166b510239C  lock 30s

GET /search → 402  "Web search with extracted page content (Brave LLM Context API)"
   resource: https://mypersonaljscloudivnad9dy-searchapi.functions.fnc.fr-par.scw.cloud/search
   eip155:10     batch-settlement  10000 atomic ($0.0100)  → 0xAAEBC1441323B8ad6Bdf6793A8428166b510239C  lock 30s
   eip155:8453   batch-settlement  10000 atomic ($0.0100)  → 0xAAEBC1441323B8ad6Bdf6793A8428166b510239C  lock 30s


## There is no other way in

This endpoint used to fork: a valid signature over `search-api:<unix ts>` was served free, and
everything else paid. That path is gone. It was justified as what kept server-side callers
working without a funded wallet, and no such caller existed — `growth-agent` never called this
endpoint, and the browser tools moved to paid fetch. `genimg` and `llm` never had one either.

The cell below sends a correctly signed owner bearer anyway, to show it now buys nothing: the
header is ignored and the answer is the same 402 an anonymous caller gets.


In [ ]:
const ts = Math.floor(Date.now() / 1000);
const message = `search-api:${ts}`;
const signature = await account.signMessage({ message });
// The shape the browser used to send, before the owner path was removed.
const token = btoa(JSON.stringify({ address: account.address, signature, message }));

const ownerRes = await fetch(`${SERVICE_URL}/fetch?url=${encodeURIComponent("https://www.fretchen.eu/")}`, {
    headers: { Authorization: `Bearer ${token}` },
});

console.log(`GET /fetch with a bearer → ${ownerRes.status}`);
console.log(
    ownerRes.status === 402
        ? "   quoted a price — the bearer was ignored, which is the point"
        : `   unexpected: ${ownerRes.status}. A bearer should buy nothing here.`,
);


## First paid call: `/fetch`

The cheap route first, at $0.001. This is the call that **opens the channel**: the SDK sees the
402, finds no channel in storage, signs a deposit, and the facilitator puts it on-chain. Expect a
real transaction hash in the settlement receipt, and a wait of a few seconds.

Everything after this one is a signature.

In [6]:
const target = "https://www.fretchen.eu/blog/36/";
const fetchRes = await fetchWithPayment(`${SERVICE_URL}/fetch?url=${encodeURIComponent(target)}`);

console.log(`GET /fetch → ${fetchRes.status}`);
if (!fetchRes.ok) {
    console.log(await fetchRes.text());
} else {
    const page = await fetchRes.json();
    console.log(`   finalUrl   : ${page.finalUrl}`);
    console.log(`   contentType: ${page.contentType}`);
    console.log(`   html       : ${page.html.length} chars`);
    console.log(`   starts     : ${JSON.stringify(page.html.slice(0, 120))}`);

    const receipt = settlement(fetchRes);
    console.log(
        `\n   settled on ${receipt?.network}: ` +
        (receipt?.transaction ? `${config.explorer}/tx/${receipt.transaction}` : "voucher only, no tx"),
    );
    const settledAsset11 = receipt?.extensions?.facilitatorFees?.info?.asset;
    if (settledAsset11) {
        console.log(`   settled asset: ${settledAsset11}`);
    }
}

GET /fetch → 200
   finalUrl   : https://www.fretchen.eu/blog/36/
   contentType: text/html


   html       : 55452 chars
   starts     : "<!DOCTYPE html>\n    <html lang=\"en\">\n      <head>\n        <link rel=\"stylesheet\" type=\"text/css\" href=\"/assets/static/la"

   settled on eip155:8453: voucher only, no tx


## Second paid call: `/search`, same channel

Ten times the price, and no transaction: the client signs a voucher against the channel that is
already open and the server commits the charge off-chain. `transaction: ""` in the receipt is what
"no second deposit" looks like.

The last block lists the channel records this directory's notebooks share. Do not expect exactly
one: Deno's `localStorage` accumulates them across notebooks and networks. The claim to check is
the receipt above — a paid call that settles with **no transaction** opened nothing, it spent on a
channel that already existed. Its `receiver` is the same `NFT_WALLET_PUBLIC_KEY` the chat endpoint
sells as, which is why a browser that had already chatted arrives here funded and sees no wallet
prompt.

In [ ]:
const searchRes = await fetchWithPayment(`${SERVICE_URL}/search?q=${encodeURIComponent("x402 payment protocol")}`);

console.log(`GET /search → ${searchRes.status}`);
if (!searchRes.ok) {
    console.log(await searchRes.text());
} else {
    const { results } = await searchRes.json();
    for (const result of results) {
        console.log(`   • ${result.title}`);
        console.log(`     ${result.url} — ${result.text.length} chars`);
    }
    const receipt = settlement(searchRes);
    console.log(`\n   settled: ${receipt?.transaction ? receipt.transaction : "voucher only, no tx"}`);
    const settledAsset13 = receipt?.extensions?.facilitatorFees?.info?.asset;
    if (settledAsset13) {
        console.log(`   settled asset: ${settledAsset13}`);
    }
}

// What the run actually proves. The storage KEY is the channelId — BatchSettlementClientContext
// itself holds only the running totals (chargedCumulativeAmount, balance, totalClaimed), no id and
// no channelConfig — so the key is what to print.
//
// Deno's localStorage is shared by every notebook in this directory, so expect records from
// sc_llm_x402_buyer.ipynb runs and from earlier networks alongside this one. "Exactly one channel"
// is therefore the wrong assertion; the right one is above, in the receipt: both paid calls
// settled with no transaction, so neither opened anything.
const channels = Array.from({ length: localStorage.length }, (_, i) => localStorage.key(i)!)
    .filter((key) => key.startsWith("x402-channel:"))
    .map((key) => ({ channelId: key.slice("x402-channel:".length), ...JSON.parse(localStorage.getItem(key)!) }));

console.log(`\n   channel records in localStorage: ${channels.length} (this directory's notebooks share one store)`);
for (const channel of channels) {
    console.log(`   ${channel.channelId}`);
    console.log(`     balance ${channel.balance ?? "?"} • charged ${channel.chargedCumulativeAmount ?? "?"} • claimed ${channel.totalClaimed ?? "?"}`);
}
console.log("\n   The one this run used is whichever advanced by 11000 — 1000 for /fetch, 10000 for /search.");

## Findings

_Fill in after a real run, as `genimg_x402_buyer.ipynb` and `sc_llm_x402_buyer.ipynb` do — each
carries a dated "observed …" section, which is how a later reader knows whether a claim was
measured or assumed._

- Observed on: _(date, network, local or deployed)_
- Deposit tx: _(hash)_
- Did `/search` settle without a transaction?
- Channels in storage after both calls:
- Anything the 402 advertised that the client did not use: